In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 04 - 声音数字化：把空气振动变成 0 和 1

**Cambridge International AS & A Level Computer Science (9618) - Section 1.2**

---

> "你听到的每一首歌、每一段语音，在计算机里都只是一长串数字。"

本节深入讲解：
- 声音的物理本质（声波）
- 模拟信号 vs 数字信号
- 采样过程的每一步
- 采样率和位深度为什么重要
- 声音文件大小计算
- 视频 = 图像 + 声音

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import sys
try:
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Libraries loaded!')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'numpy', '-q'])
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Installed & loaded!')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10,6),
    'font.sans-serif': ['WenQuanYi Zen Hei','Noto Sans CJK SC','DejaVu Sans'], 'axes.unicode_minus': False})


---
## 1. 声音的物理本质

### 1.1 声音是什么？

声音是 **空气（或其他介质）中粒子的振动**，以波的形式传播。

- **频率 (Frequency)**：每秒振动次数（单位：Hz）
  - 低频 = 低音（如低音炮的轰鸣）
  - 高频 = 高音（如蚊子的嗡嗡声）
  - 人耳范围：约 20 Hz ~ 20,000 Hz

- **振幅 (Amplitude)**：振动的幅度
  - 振幅大 = 声音响
  - 振幅小 = 声音轻

- **波长 (Wavelength)**：一个完整波形的长度

### 1.2 模拟信号 vs 数字信号

| | 模拟信号 (Analogue) | 数字信号 (Digital) |
|:---|:---|:---|
| **特点** | 连续的，可以取任意值 | 离散的，只能取有限值 |
| **例子** | 声波、温度、光线 | 计算机数据、CD 音乐 |
| **优点** | 完全保真，无信息丢失 | 易于存储、复制、传输 |
| **缺点** | 容易受噪声干扰，难复制 | 采样过程中会丢失细节 |

**核心问题：** 怎么把连续的声波变成计算机能存储的离散数字？

答案就是 **采样 (Sampling)**。

In [ ]:
# 可视化：模拟波 vs 数字化
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

t = np.linspace(0, 1, 1000)
wave = np.sin(2 * np.pi * 3 * t) * 0.8

# 1. Continuous analogue wave
axes[0].plot(t, wave, 'b-', linewidth=2)
axes[0].fill_between(t, wave, alpha=0.1, color='blue')
axes[0].set_title('Analogue Signal\n(Continuous)', fontsize=13, fontweight='bold', color='blue')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# 2. Sampled points
n_samples = 20
t_s = np.linspace(0, 1, n_samples)
w_s = np.sin(2 * np.pi * 3 * t_s) * 0.8
axes[1].plot(t, wave, 'b-', alpha=0.2, linewidth=1)
axes[1].vlines(t_s, 0, w_s, colors='red', linewidth=2, alpha=0.5)
axes[1].plot(t_s, w_s, 'ro', markersize=8, zorder=5)
axes[1].set_title('Sampling\n(Measuring at intervals)', fontsize=13, fontweight='bold', color='red')
axes[1].set_xlabel('Time')
axes[1].grid(True, alpha=0.3)

# 3. Digital reconstruction
axes[2].step(t_s, w_s, 'g-', linewidth=2, where='mid')
axes[2].plot(t_s, w_s, 'go', markersize=6, zorder=5)
axes[2].set_title('Digital Signal\n(Discrete values)', fontsize=13, fontweight='bold', color='green')
axes[2].set_xlabel('Time')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_ylim(-1.2, 1.2)

plt.suptitle('Analogue to Digital Conversion (ADC)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Step 1: Start with continuous analogue sound wave')
print('Step 2: Measure (sample) the amplitude at regular intervals')
print('Step 3: Store each measurement as a digital number')
print('Step 4: The digital signal approximates the original')

---
## 2. 采样率 (Sampling Rate)

### 2.1 什么是采样率？

**采样率** = 每秒钟采集多少个样本点

单位：**Hz**（赫兹）或 **kHz**（千赫兹）

| 采样率 | 用途 | 每秒样本数 |
|:---|:---|---:|
| 8,000 Hz | 电话通话 | 8,000 |
| 22,050 Hz | AM 广播级 | 22,050 |
| 44,100 Hz | CD 音质 | 44,100 |
| 48,000 Hz | DVD / 专业音频 | 48,000 |
| 96,000 Hz | 录音棚高清 | 96,000 |
| 192,000 Hz | 发烧级音频 | 192,000 |

### 2.2 为什么 CD 用 44,100 Hz？

这和 **奈奎斯特定理 (Nyquist Theorem)** 有关：

> 要完美还原一个频率为 f 的信号，采样率必须至少是 **2f**。

人耳能听到的最高频率约 20,000 Hz，所以：
- 最低采样率 = 2 x 20,000 = 40,000 Hz
- CD 标准选了 44,100 Hz（留一点余量）

**这就是为什么 44,100 Hz 成了标准！** 不是随便选的，而是有数学依据的。

In [ ]:
# 采样率对比：直观感受差异
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

t_fine = np.linspace(0, 0.5, 2000)
# A complex wave (like real audio)
wave = (0.5 * np.sin(2*np.pi*5*t_fine) +
        0.3 * np.sin(2*np.pi*13*t_fine) +
        0.2 * np.sin(2*np.pi*27*t_fine))

rates = [6, 12, 24, 48, 96, 200]
titles = [
    '6 samples\n(Way too few!)',
    '12 samples\n(Still losing detail)',
    '24 samples\n(Getting better)',
    '48 samples\n(Pretty good)',
    '96 samples\n(Very accurate)',
    '200 samples\n(Nearly perfect)',
]

for ax, n, title in zip(axes.flat, rates, titles):
    ax.plot(t_fine, wave, 'b-', alpha=0.25, linewidth=1, label='Original')
    t_s = np.linspace(0, 0.5, n)
    w_s = (0.5*np.sin(2*np.pi*5*t_s) + 0.3*np.sin(2*np.pi*13*t_s) + 0.2*np.sin(2*np.pi*27*t_s))
    ax.vlines(t_s, min(wave)-0.1, w_s, colors='red', alpha=0.3, linewidth=1)
    ax.plot(t_s, w_s, 'ro', markersize=3)
    ax.step(t_s, w_s, 'r-', linewidth=1.5, alpha=0.7, where='mid', label='Digital')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(min(wave)-0.3, max(wave)+0.3)
    ax.grid(True, alpha=0.2)

plt.suptitle('Sampling Rate: More Samples = More Accurate Reproduction',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice how the red staircase gets closer to the blue wave')
print('as we add more sample points.')
print()
print('In real audio:')
print('  8,000 Hz  = phone quality (sounds robotic)')
print('  44,100 Hz = CD quality (sounds natural)')
print('  96,000 Hz = studio quality (audiophile grade)')

---
## 3. 采样分辨率 / 位深度 (Bit Depth)

### 3.1 什么是位深度？

位深度 = 每个采样值用多少 bit 来记录

它决定了振幅能被分成 **多少个级别**。

| 位深度 | 级别数 | 用途 |
|:---|:---|:---|
| 4 bits | 2^4 = 16 | 非常粗糙 |
| 8 bits | 2^8 = 256 | 电话级 |
| 16 bits | 2^16 = 65,536 | CD 音质 |
| 24 bits | 2^24 = 16,777,216 | 录音棚级 |

### 3.2 为什么级别数很重要？

想象你要记录一个音量为 0.73 的声音：
- 4 位 (16 级): 最接近的是 0.75 或 0.6875，有误差
- 16 位 (65536 级): 最接近的是 0.729996...，误差极小

**位深度越高 = 级别越多 = 量化误差越小 = 声音越精确**

这个误差叫做 **量化噪声 (Quantization Noise)**——你有时在低质量音频中听到的那种"沙沙"声。

In [ ]:
# 位深度对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

t = np.linspace(0, 1, 500)
wave = np.sin(2 * np.pi * 3 * t) * 0.45 + 0.5

for ax, bits, title in zip(axes.flat, [2, 4, 8, 16],
    ['2-bit: 4 levels', '4-bit: 16 levels', '8-bit: 256 levels', '16-bit: 65536 levels']):
    n_levels = 2 ** bits

    # Original
    ax.plot(t, wave, 'b-', alpha=0.3, linewidth=1, label='Original')

    # Quantized
    q = np.round(wave * (n_levels - 1)) / (n_levels - 1)
    ax.step(t, q, 'r-', linewidth=2, label=f'Quantized', where='mid')

    # Show levels as horizontal lines
    if n_levels <= 16:
        for lv in np.linspace(0, 1, n_levels):
            ax.axhline(y=lv, color='gray', linewidth=0.5, alpha=0.4, linestyle='--')

    # Show error
    error = np.mean(np.abs(wave - q))
    ax.set_title(f'{title}\nAvg error: {error:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('Time'); ax.set_ylabel('Amplitude')

plt.suptitle('Bit Depth: More Levels = Less Quantization Error', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('2-bit:  Average error = 0.0833 (VERY noticeable)')
print('4-bit:  Average error = 0.0208 (noticeable)')
print('8-bit:  Average error = 0.0013 (barely noticeable)')
print('16-bit: Average error = 0.000005 (impossible to hear)')
print()
print('Each extra bit HALVES the quantization error!')

---
## 4. 声音文件大小计算

### 核心公式

$$\boxed{\text{文件大小 (bits)} = \text{采样率} \times \text{位深度} \times \text{声道数} \times \text{时长(秒)}}$$

$$\text{文件大小 (bytes)} = \frac{\text{文件大小 (bits)}}{8}$$

**声道数：**
- Mono（单声道）= 1
- Stereo（立体声）= 2（左右各一个声道）

### 为什么立体声文件是单声道的两倍？
因为要同时记录左耳和右耳听到的声音，相当于两份数据。

In [ ]:
# Sound file size calculator
def calc_sound(sec, rate, bits, channels=1):
    total_samples = rate * sec * channels
    total_bits = total_samples * bits
    total_bytes = total_bits / 8
    mb = total_bytes / 1e6

    ch_name = 'Mono' if channels == 1 else 'Stereo'
    minutes = sec // 60
    secs = sec % 60
    time_str = f'{minutes}m {secs}s' if minutes else f'{secs}s'

    print('=' * 60)
    print(f'  Sound File Size Calculator')
    print('=' * 60)
    print(f'  Duration:    {time_str} ({sec} seconds)')
    print(f'  Sample rate: {rate:,} Hz ({rate/1000:.1f} kHz)')
    print(f'  Bit depth:   {bits} bits ({2**bits:,} levels)')
    print(f'  Channels:    {channels} ({ch_name})')
    print()
    print(f'  Step 1: Samples = {rate:,} x {sec} x {channels} = {total_samples:,}')
    print(f'  Step 2: Bits    = {total_samples:,} x {bits} = {total_bits:,}')
    print(f'  Step 3: Bytes   = {total_bits:,} / 8 = {total_bytes:,.0f}')
    print(f'  Step 4: MB      = {mb:.2f} MB')
    print('=' * 60)
    return total_bytes

print('Example 1: CD quality 3-minute song')
calc_sound(180, 44100, 16, 2)

print('\nExample 2: Phone call 1 minute')
calc_sound(60, 8000, 8, 1)

print('\nExample 3: Studio recording 1 minute')
calc_sound(60, 96000, 24, 2)

---
## 5. 视频 = 图像序列 + 音频

视频本质上就是 **快速播放的一系列图片（帧）+ 同步的音频轨道**。

### 帧率 (Frame Rate)

| 帧率 | 名称 | 用途 |
|:---|:---|:---|
| 24 fps | 电影标准 | 电影院 |
| 25 fps | PAL | 欧洲/中国电视 |
| 30 fps | NTSC | 美国/日本电视 |
| 60 fps | 高帧率 | 游戏、体育直播 |
| 120 fps | 超高帧率 | 高端游戏显示器 |

### 视频文件大小 = 视频轨 + 音频轨

$$\text{视频轨} = W \times H \times depth \times fps \times seconds \div 8$$
$$\text{音频轨} = rate \times bits \times channels \times seconds \div 8$$
$$\text{总大小} = \text{视频轨} + \text{音频轨}$$

In [ ]:
# Video file size calculator
def calc_video(w, h, depth, fps, sec, audio_rate=44100, audio_bits=16, audio_ch=2):
    video_bytes = w * h * depth * fps * sec / 8
    audio_bytes = audio_rate * audio_bits * audio_ch * sec / 8
    total = video_bytes + audio_bytes

    print('=' * 60)
    print(f'  Video File Size Calculator (uncompressed)')
    print('=' * 60)
    print(f'  Video: {w}x{h}, {depth}-bit, {fps} fps, {sec}s')
    print(f'  Audio: {audio_rate}Hz, {audio_bits}-bit, {"stereo" if audio_ch==2 else "mono"}')
    print()
    print(f'  Video track: {video_bytes/1e9:.2f} GB')
    print(f'  Audio track: {audio_bytes/1e6:.1f} MB')
    print(f'  Total:       {total/1e9:.2f} GB')
    print('=' * 60)
    return total

print('1 minute of uncompressed video:')
print()
print('--- Full HD (1920x1080, 24fps) ---')
calc_video(1920, 1080, 24, 24, 60)
print()
print('--- 4K (3840x2160, 30fps) ---')
calc_video(3840, 2160, 24, 30, 60)
print()
print('--- 4K Gaming (3840x2160, 60fps) ---')
calc_video(3840, 2160, 24, 60, 60)

print()
print('This is why video compression (MP4, H.264, H.265) is ESSENTIAL!')
print('A 2-hour 4K movie uncompressed would be ~3 TERABYTES!')
print('With H.265 compression it is about 5-10 GB.')

---
## 6. 练习题

**计算题：**
1. 一首歌 4 分钟，44,100 Hz，16位，立体声。文件多大？
2. 语音录音 8,000 Hz，8位，单声道，1小时。需要多少空间？
3. 一段 30 秒的 Full HD 视频 (1920x1080, 24-bit, 25fps) + CD音质音频，总共多大？

**理解题：**
4. 为什么更高的采样率能产生更好的音质？用奈奎斯特定理解释。
5. 说出 3 种减小声音文件大小的方法（不使用压缩算法）。
6. 为什么 CD 音质是 44,100 Hz 而不是 40,000 Hz？
7. 什么是量化噪声？怎么减少它？
8. 为什么 1 分钟未压缩 4K 视频比 1 分钟 CD 音乐大得多？

In [ ]:
# Answers
print('Q1:'); calc_sound(240, 44100, 16, 2)
print('\nQ2:'); calc_sound(3600, 8000, 8, 1)
print('\nQ3:'); calc_video(1920, 1080, 24, 25, 30)

print("""
Q4: Nyquist theorem says sampling rate must be >= 2x the highest frequency.
    Higher sampling rate captures higher frequencies and more detail.
    44,100 Hz captures up to ~22,050 Hz (above human hearing limit of ~20kHz).
    8,000 Hz only captures up to 4,000 Hz (phone quality, loses high frequencies).

Q5: Three ways to reduce sound file size without compression:
    1. Lower sampling rate (44100 -> 22050 Hz) - halves file size
    2. Lower bit depth (16-bit -> 8-bit) - halves file size
    3. Use mono instead of stereo - halves file size
    (Also: shorten the recording duration)

Q6: 40,000 Hz is the theoretical minimum (2 x 20kHz).
    44,100 Hz adds ~10% headroom for the anti-aliasing filter
    to work properly. The filter needs a transition band
    between 20kHz (pass) and 22.05kHz (stop).

Q7: Quantization noise is the error between the actual amplitude
    and the nearest available digital level. It sounds like a
    faint hissing or buzzing. Reduce it by using higher bit depth
    (more levels = smaller gaps between levels = less error).

Q8: Video = many images per second + audio.
    1 FHD frame = 1920x1080x24/8 = ~6 MB
    25 frames/sec x 60 sec = 1500 frames
    1500 x 6 MB = ~9 GB for video alone!
    CD audio for 1 min = ~10 MB
    So video is ~900x larger than audio!""")

---
**Next:** [05 - 文件压缩](05_文件压缩.ipynb)